<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Galton_Board_and_the_Central_Limit_Theorem_Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Galton Board Simulation

This notebook contains a high-fidelity physics simulation of a Galton Board (also known as a Bean Machine).

### Features:
- **Realistic Physics**: Gravity-based acceleration and peg collisions with energy loss (restitution).
- **Visual Stacking**: Particles physically stack within bins at the bottom.
- **Statistical Analysis**: Real-time histogram and Normal Distribution curve overlay demonstrating the Central Limit Theorem.
- **Multi-filling**: The simulation runs through multiple cycles of ball releases.

### Parameters:
You can modify parameters like `TOTAL_BALLS`, `PEG_ROWS`, and `GRAVITY` directly in the code cell below.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.stats import norm
from IPython.display import HTML
import matplotlib

"""
Galton Board Simulation Module - Mobile Optimized
------------------------------------------------
Simulates particles falling through a triangular grid of pegs,
bouncing realistically, and stacking in bins.

Optimized for portrait (upright) mobile viewing with a vertical aspect ratio.
"""

# Optimization for file size and memory
matplotlib.rcParams['animation.embed_limit'] = 100.0

# --- EDITABLE PARAMETERS ---
TOTAL_BALLS = 500
PEG_ROWS = 12
GRAVITY = 0.3
BOUNCE_COEFF = 0.35
PEG_RADIUS = 0.1
BALL_COLOR = '#FFFFFF'
PEG_COLOR = '#00FFFF'
BIN_COLOR = '#008080'
FPS = 30
DURATION_SEC = 40
TOTAL_FRAMES = FPS * DURATION_SEC
# ---------------------------

plt.style.use('dark_background')

# Dramatic change to aspect ratio: 9:16 vertical layout (approx 6.75x12 inches)
fig, (ax_board, ax_hist) = plt.subplots(2, 1, figsize=(6.75, 12), gridspec_kw={'height_ratios': [1.8, 1]})
fig.suptitle('Galton Board', fontsize=22, color='white', y=0.98)

# Setup Axes
ax_board.set_xlim(-PEG_ROWS/2 - 1, PEG_ROWS/2 + 1)
ax_board.set_ylim(-5, PEG_ROWS + 2)
ax_board.axis('off')

ax_hist.set_xlim(-PEG_ROWS/2 - 1, PEG_ROWS/2 + 1)
ax_hist.set_ylim(0, TOTAL_BALLS / 8)
ax_hist.set_title('Live Distribution', fontsize=16)
ax_hist.tick_params(axis='both', which='major', labelsize=12)

# Static Elements
pegs = []
for r in range(PEG_ROWS):
    y = PEG_ROWS - r
    xs = np.linspace(-r * 0.5, r * 0.5, r + 1)
    for x in xs: pegs.append((x, y))
pegs = np.array(pegs)
ax_board.scatter(pegs[:, 0], pegs[:, 1], color=PEG_COLOR, s=20, alpha=0.5, zorder=2)

bin_centers = np.arange(-PEG_ROWS/2, PEG_ROWS/2 + 1, 1)
ax_board.plot([-PEG_ROWS/2 - 1, PEG_ROWS/2 + 1], [-0.5, -0.5], color=BIN_COLOR, lw=3)

bins_counts = {bc: 0 for bc in bin_centers}
bars = ax_hist.bar(bin_centers, [0]*len(bin_centers), color=BALL_COLOR, alpha=0.3)
line_curve, = ax_hist.plot([], [], color='#FF00FF', lw=3)

# State variables
active_balls = []
landed_points_data = []
balls_released_this_cycle = 0
cycle_limit = TOTAL_BALLS // 2

ball_dots = [ax_board.plot([], [], 'o', color=BALL_COLOR, markersize=5)[0] for _ in range(30)]
static_balls = []

def update(frame):
    global balls_released_this_cycle, bins_counts, landed_points_data, static_balls

    if balls_released_this_cycle >= cycle_limit and not active_balls:
        balls_released_this_cycle = 0
        bins_counts = {bc: 0 for bc in bin_centers}
        landed_points_data = []
        for sb in static_balls: sb.remove()
        static_balls = []

    if frame % 3 == 0 and balls_released_this_cycle < cycle_limit:
        active_balls.append({'pos': [np.random.uniform(-0.1, 0.1), PEG_ROWS + 0.5], 'vel': [0, 0]})
        balls_released_this_cycle += 1

    for ball in active_balls[:]:
        ball['vel'][1] -= GRAVITY * 0.15
        ball['pos'][0] += ball['vel'][0]
        ball['pos'][1] += ball['vel'][1]

        for px, py in pegs:
            dist = np.sqrt((ball['pos'][0]-px)**2 + (ball['pos'][1]-py)**2)
            if dist < PEG_RADIUS + 0.1:
                nx, ny = (ball['pos'][0]-px)/dist, (ball['pos'][1]-py)/dist
                dot = ball['vel'][0]*nx + ball['vel'][1]*ny
                ball['vel'][0] = (ball['vel'][0] - 2*dot*nx) * BOUNCE_COEFF + np.random.uniform(-0.04, 0.04)
                ball['vel'][1] = (ball['vel'][1] - 2*dot*ny) * BOUNCE_COEFF
                ball['pos'][0] = px + 0.21 * nx
                ball['pos'][1] = py + 0.21 * ny

        if ball['pos'][1] <= -0.5:
            bc = bin_centers[np.argmin(np.abs(bin_centers - ball['pos'][0]))]
            stack_y = -0.7 - (bins_counts[bc] * 0.12)
            p, = ax_board.plot(bc, stack_y, 'o', color=BALL_COLOR, markersize=3, alpha=0.5)
            static_balls.append(p)
            bins_counts[bc] += 1
            landed_points_data.append(bc)
            active_balls.remove(ball)
            for i, center in enumerate(bin_centers):
                bars[i].set_height(bins_counts[center])
            if len(landed_points_data) > 10:
                mu, std = np.mean(landed_points_data), np.std(landed_points_data)
                if std > 0:
                    x_s = np.linspace(-PEG_ROWS/2-1, PEG_ROWS/2+1, 50)
                    line_curve.set_data(x_s, norm.pdf(x_s, mu, std) * len(landed_points_data))

    for i, dot in enumerate(ball_dots):
        if i < len(active_balls):
            dot.set_data([active_balls[i]['pos'][0]], [active_balls[i]['pos'][1]])
        else:
            dot.set_data([], [])

    return list(bars) + [line_curve] + ball_dots + static_balls

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
ani = FuncAnimation(fig, update, frames=TOTAL_FRAMES, interval=1000/FPS, blit=True)
plt.close(fig)
HTML(ani.to_html5_video())